# Question 9: String Matching in Search Engines

## Introduction

A search engine must locate query patterns inside large document collections. This notebook implements **Naive String Matching** and the **Knuth-Morris-Pratt (KMP)** algorithm, then stress-tests both against Python's own `str.find()` across 500 randomized cases — including the classic repeating-pattern cases that expose bugs in a naive/incorrect KMP implementation.

## (a) Naive String Matching

Slides the pattern over the text one position at a time, comparing character by character at each position and stopping early on the first mismatch.

In [1]:
def naive_search(text, pattern):
    n, m = len(text), len(pattern)
    for i in range(n - m + 1):
        j = 0
        while j < m and text[i + j] == pattern[j]:
            j += 1
        if j == m:
            return i
    return -1

print(naive_search("Advanced Algorithms", "Algorithms"))
print(naive_search("hello world", "xyz"))

9
-1


## (b) Knuth-Morris-Pratt (KMP) Algorithm

KMP avoids re-examining characters it has already matched by precomputing a **Longest Prefix-Suffix (LPS)** array for the pattern, which tells it how far to "skip back" on a mismatch instead of restarting from scratch.

In [2]:
def compute_lps(pattern):
    lps = [0] * len(pattern)
    length = 0
    i = 1
    while i < len(pattern):
        if pattern[i] == pattern[length]:
            length += 1
            lps[i] = length
            i += 1
        else:
            if length != 0:
                length = lps[length - 1]
            else:
                lps[i] = 0
                i += 1
    return lps

def kmp_search(text, pattern):
    if pattern == "":
        return 0
    lps = compute_lps(pattern)
    i = j = 0
    while i < len(text):
        if text[i] == pattern[j]:
            i += 1
            j += 1
            if j == len(pattern):
                return i - j
        else:
            if j != 0:
                j = lps[j - 1]
            else:
                i += 1
    return -1

print(kmp_search("Advanced Algorithms", "Algorithms"))
print(kmp_search("hello world", "xyz"))

9
-1


In [3]:
# ---- Tricky repeating-pattern cases (these expose bugs in a naive/incorrect KMP implementation) ----
tricky_cases = [
    ("AAAAAAAAAB", "AAAAB"),
    ("ABABDABACDABABCABAB", "ABABCABAB"),
    ("AAAAAAAA", "AAA"),
    ("MISSISSIPPI", "ISSI"),
]
for text, pattern in tricky_cases:
    expected = text.find(pattern)
    naive_result = naive_search(text, pattern)
    kmp_result = kmp_search(text, pattern)
    assert naive_result == expected, f"naive_search wrong for {pattern!r} in {text!r}"
    assert kmp_result == expected, f"kmp_search wrong for {pattern!r} in {text!r}"
    print(f"text={text!r:24} pattern={pattern!r:12} -> index {expected} (both algorithms agree ✔)")

text='AAAAAAAAAB'             pattern='AAAAB'      -> index 5 (both algorithms agree ✔)
text='ABABDABACDABABCABAB'    pattern='ABABCABAB'  -> index 10 (both algorithms agree ✔)
text='AAAAAAAA'               pattern='AAA'        -> index 0 (both algorithms agree ✔)
text='MISSISSIPPI'            pattern='ISSI'       -> index 1 (both algorithms agree ✔)


In [4]:
# ---- Stress test: 500 randomized cases cross-validated against Python's str.find() ----
import random
random.seed(1)
alphabet = "ab"   # small alphabet maximizes the chance of repeating sub-patterns, the hardest case for KMP
trials = 500
for _ in range(trials):
    text = ''.join(random.choice(alphabet) for _ in range(random.randint(1, 40)))
    pattern = ''.join(random.choice(alphabet) for _ in range(random.randint(1, 8)))
    expected = text.find(pattern)
    assert naive_search(text, pattern) == expected
    assert kmp_search(text, pattern) == expected

print(f"{trials}/{trials} randomized trials PASSED for both algorithms ✔ (cross-validated against str.find)")

500/500 randomized trials PASSED for both algorithms ✔ (cross-validated against str.find)


## (c) Comparison of Algorithms

| Algorithm | Time Complexity | Space Complexity |
|---|---|---|
| Naive Search | O(n·m) | O(1) |
| KMP | O(n + m) | O(m) for the LPS array |

Where n = text length, m = pattern length.

In [5]:
import time

# Worst case for naive search: text and pattern both highly repetitive (e.g. "AAAA...A")
# This forces naive_search to compare almost the full pattern length at *every* starting
# position before failing, which is exactly the O(n*m) pathological case.
n = 20000
text = "A" * n
pattern = "A" * 200 + "B"   # never matches -> forces near worst-case comparisons

start = time.perf_counter()
naive_search(text, pattern)
t_naive = time.perf_counter() - start

start = time.perf_counter()
kmp_search(text, pattern)
t_kmp = time.perf_counter() - start

print(f"Naive Search time: {t_naive:.6f}s")
print(f"KMP time:          {t_kmp:.6f}s")
print(f"KMP was {t_naive/t_kmp:.1f}x faster on this adversarial repetitive input")
assert t_kmp < t_naive

Naive Search time: 0.384214s
KMP time:          0.005668s
KMP was 67.8x faster on this adversarial repetitive input


**A note on benchmarking honesty:** an earlier draft of this benchmark implemented naive search using Python's built-in slice comparison (`text[i:i+m] == pattern`), which silently delegates the character comparison to a heavily-optimized C routine. That made the "naive" algorithm look *faster* than KMP in wall-clock time on this input — a misleading artifact of implementation, not a reflection of the O(n·m) vs O(n+m) complexity difference being compared. The version benchmarked above performs the comparison itself, character by character in pure Python, exactly as the naive algorithm is defined — giving an honest, apples-to-apples comparison where KMP's asymptotic advantage is actually visible.

## (d) Performance Analysis

The Naive algorithm re-examines characters it has already compared whenever a mismatch occurs, which becomes especially costly on **repetitive text** (e.g. `"AAAA...A"`) — exactly the adversarial case benchmarked above, where it is roughly two orders of magnitude slower. KMP's LPS table lets it "remember" partial matches and skip redundant comparisons, giving it linear O(n + m) time even in this worst case.

**Conclusion:** for a search engine indexing large, naturally repetitive text (HTML markup, repeated boilerplate, DNA-like or log-like data), KMP's linear worst-case guarantee makes it the safer production choice over naive search.